In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

from datetime import datetime
import sys
import os
from pathlib import Path
from typing import Any, Tuple, List, Dict
from dotmap import DotMap
import json

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib as mpl
import plotly
import plotly.express as px
import plotly.graph_objs as go
import plotly.io as pio
from plotly.subplots import make_subplots
import statsmodels.api as sm
from scipy.stats import ttest_rel, binomtest, wilcoxon
from tqdm.notebook import tqdm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from swc.aeon.io import api as aeon_api
from swc.aeon.io import reader as aeon_reader
from aeon.schema.schemas import social02, exp02

cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)
from data_io_utils import save_all_experiment_data, load_data_from_parquet

# Definitions

In [ ]:
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
save_dir = Path("/ceph/aeon/aeon/code/scratchpad/anaya/methods_paper_figs")
os.makedirs(data_dir, exist_ok=True)
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas
light_off, light_on = 7, 20  # 7am to 7pm
fps = 50

In [ ]:
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 11:00:00', "social_end": '2024-07-06 13:00:00', "postsocial_start": '2024-07-07 16:00:00', "postsocial_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "presocial_start": '2024-06-08 19:00:00', "presocial_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 12:00:00', "social_end": '2024-07-03 14:00:00', "postsocial_start": '2024-07-04 11:00:00', "postsocial_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "presocial_start": '2024-08-16 17:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 11:00:00', "social_end": '2024-09-09 13:00:00', "postsocial_start": '2024-09-09 18:00:00', "postsocial_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "presocial_start": '2024-08-16 15:00:00', "presocial_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 01:00:00', "postsocial_start": '2024-09-09 15:00:00', "postsocial_end": '2024-09-22 16:00:00'}
]
experiment = experiments[-2]

In [ ]:
def load_experiment_data(data_dir, experiment=None, periods=None, 
                        data_types=['rfid', 'position'], trim_days=None):
    """
    Load all data types for specified periods of an experiment.
    
    Parameters:
    - experiment: experiment dict with period start/end times
    - periods: list of periods to load
    - data_types: list of data types to load
    - data_dir: directory containing data files
    - trim_days: Optional number of days to trim from start (None = no trim)
    
    Returns:
    - Dictionary containing dataframes for each period/data type combination
    """
    
    result = {}

    if periods is None:
        periods = [None]
    
    for period in periods:
        for data_type in data_types:
            print(f"Loading {period} {data_type} data...")
            
            # Load data
            if experiment is not None:
                experiment_name = experiment["name"]
            else:
                experiment_name = None
            df = load_data_from_parquet(
                experiment_name=experiment_name,
                period=period,
                data_type=data_type,
                data_dir=data_dir,
                set_time_index=(data_type == 'position')
            )
            
            # Trim if requested
            if trim_days is not None and len(df) > 0:
                if data_type == 'rfid':
                    start_time = df['chunk_start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['chunk_start'] < end_time]
                if data_type == 'foraging':
                    start_time = df['start'].min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df[df['start'] < end_time]
                if data_type == 'position':
                    start_time = df.index.min()
                    end_time = start_time + pd.Timedelta(days=trim_days)
                    df = df.loc[df.index < end_time]
                
                print(f"  Trimmed to {trim_days} days: {len(df)} records")
            
            # Store in result
            key = f"{period}_{data_type}"
            result[key] = df
            
            # For position data, handle duplicates
            if data_type == 'position' and len(df) > 0:
                original_len = len(df)
                df = df.reset_index()
                df = df.drop_duplicates(subset=['time', 'identity_name'])
                df = df.set_index('time')
                result[key] = df
                if len(df) < original_len:
                    print(f"  Removed duplicates: {original_len} -> {len(df)}")
    
    return result

In [ ]:
"""Utilities for tracking data processing."""

NUM_IDS = 2  # Number of identities expected in the tracking data
MAX_DIST_BETWEEN_FRAMES = 90  # Maximum distance between frames
MIN_INTER_SUBJ_DIST = 100  # Subjects must be at least this far apart
MAX_SWAP_COST = 100  # Maximum cost distance to consider a swap


def filter_valid_points(df: pd.DataFrame, region_df: pd.DataFrame) -> pd.DataFrame:
    """Filter points in the DataFrame that are within the arena or nest region.

    Args:
        df (pandas.DataFrame): DataFrame containing tracking data with columns 'x' and 'y'.
        region_df (pandas.DataFrame): DataFrame containing arena region data with columns:
            - region_name: name of the region (e.g., 'ArenaCenter', 'ArenaOuterRadius', 'NestRegion')
            - region_data: dictionary containing region-specific data

    Returns:
        pandas.DataFrame: DataFrame with only valid (in-arena or in-nest) points

    """

    def get_region_value(region_name):
        mask = region_df.index.get_level_values("region_name") == region_name
        if mask.any():
            return region_df.loc[mask, "region_data"].iloc[0]  # type: ignore
        return None

    # --- Extract arena and nest geometry ---
    arena_center = get_region_value("ArenaCenter")
    arena_outer_radius = get_region_value("ArenaOuterRadius")
    nest_region = get_region_value("NestRegion")

    if arena_center is None or arena_outer_radius is None:
        raise ValueError("Could not find ArenaCenter or ArenaOuterRadius in region data")

    center_x = float(arena_center["X"])
    center_y = float(arena_center["Y"])
    outer_radius = float(arena_outer_radius) + 10  # extra padding

    coords = df[["x", "y"]].to_numpy()
    dx = coords[:, 0] - center_x
    dy = coords[:, 1] - center_y
    dist2 = dx**2 + dy**2
    inside_arena = dist2 <= outer_radius**2

    inside_nest = np.zeros(len(df), dtype=bool)
    if nest_region and "ArrayOfPoint" in nest_region:
        nest_coords = np.array([(float(p["X"]), float(p["Y"])) for p in nest_region["ArrayOfPoint"]])
        if len(nest_coords) > 0:
            x_min, x_max = nest_coords[:, 0].min() - 10, nest_coords[:, 0].max() + 10
            y_min, y_max = nest_coords[:, 1].min() - 10, nest_coords[:, 1].max() + 10
            inside_nest = (
                (coords[:, 0] >= x_min)
                & (coords[:, 0] <= x_max)
                & (coords[:, 1] >= y_min)
                & (coords[:, 1] <= y_max)
            )

    return df.loc[inside_arena | inside_nest]


def clean_swaps(df: pd.DataFrame, region_df: pd.DataFrame) -> pd.DataFrame:
    """Swap correction for dual mouse tracking.

    Filters out-of-bounds points first,
    then does identity assignment with majority voting to fix track swaps.

    - Pre-cleaning: removes points outside arena/nest bounds
    - Early frames: filled with raw data before first complete observation
    - Swap detection: uses frame-to-frame distance minimization
    - Identity correction: SLEAP-style majority vote within continuous segments of cleaning
    - Swap marking: sets identity_likelihood=NaN on locally swapped frames

    Args:
        df (pandas.DataFrame): DataFrame containing tracking data with columns 'x' and 'y'.
        region_df (pandas.DataFrame): DataFrame containing arena region data with columns:
            - region_name: name of the region (e.g., 'ArenaCenter', 'ArenaOuterRadius', 'NestRegion')
            - region_data: dictionary containing region-specific data

    Returns:
        pandas.DataFrame: DataFrame with cleaned tracking data having the same structure as
            the input but corrected x,y coords and identity_likelihood=NaN on locally swapped frames

    Examples:
        To retrieve region_df from the database:

        >>> active_region_query = acquisition.EpochConfig.ActiveRegion & (acquisition.Chunk & chunk_key)
        >>> region_df = active_region_query.fetch(format="frame")

    """
    # Select only points within the arena or nest region
    df = filter_valid_points(df, region_df)

    # Swap correction
    # 1) setup data for processing
    df = df.sort_index()
    ids = sorted(df["identity_name"].unique())  # enforce order
    if len(ids) != NUM_IDS:
        raise ValueError(f"Expected exactly two identities, found: {ids}")

    # 2) Prep for merge later
    df2 = df.reset_index()
    time_col = df2.columns[0]  # timestamp column name

    # 3) Reshape to 2×T arrays
    wide = df2.pivot(index=time_col, columns="identity_name", values=["x", "y"])
    times = wide.index.values
    T = len(times)
    x_raw = wide["x"][ids].to_numpy().T
    y_raw = wide["y"][ids].to_numpy().T

    # 4) Init cleaned arrays + swap tracking
    x_clean = np.full_like(x_raw, np.nan)
    y_clean = np.full_like(y_raw, np.nan)
    swapped_flags = np.zeros(T, dtype=bool)

    # 5) Find first complete frame and keep raw data before it
    valid = np.isfinite(x_raw).all(axis=0)
    if not valid.any():
        raise RuntimeError("No frame with both subjects present")
    first_i = np.argmax(valid)
    if first_i > 0:
        x_clean[:, :first_i] = x_raw[:, :first_i]
        y_clean[:, :first_i] = y_raw[:, :first_i]

    # 6) Local-segment tracking
    # Initialize on first full-detect frame
    seg_start = first_i
    votes_same = 1
    votes_swap = 0
    last_x = x_raw[:, first_i].copy()
    last_y = y_raw[:, first_i].copy()
    x_clean[:, first_i] = last_x
    y_clean[:, first_i] = last_y

    # --- Helper functions ---
    def _flush_segment(start, end, votes_same, votes_swap):
        """Flush the current segment and apply local vote."""
        if votes_swap > votes_same:
            x_clean[:, start:end] = x_clean[::-1, start:end]
            y_clean[:, start:end] = y_clean[::-1, start:end]
            swapped_flags[start:end] = ~swapped_flags[start:end]

    def _assign_single_detection(t, src_idx, dest_idx):
        """Assign values from src_idx to dest_idx at time t."""
        x_clean[dest_idx, t] = x_raw[src_idx, t]
        y_clean[dest_idx, t] = y_raw[src_idx, t]
        last_x[dest_idx] = x_raw[src_idx, t]
        last_y[dest_idx] = y_raw[src_idx, t]

    def _assign_full_frame(t, x_vals, y_vals):
        """Assign full frame values to both identities at time t."""
        x_clean[:, t] = x_vals
        y_clean[:, t] = y_vals
        last_x[:] = x_vals
        last_y[:] = y_vals

    def _update_votes(t):
        """Determine if a swap occurred at time t and update vote counts accordingly."""
        if np.allclose(x_raw[:, t], x_clean[:, t], equal_nan=True) and np.allclose(
            y_raw[:, t], y_clean[:, t], equal_nan=True
        ):
            nonlocal votes_same
            votes_same += 1
        else:
            nonlocal votes_swap
            votes_swap += 1

    for t in tqdm(range(first_i + 1, T), desc="Cleaning frames"):
        present = np.isfinite(x_raw[:, t])
        n_det = present.sum()

        # zero detections → drop both
        if n_det == 0:
            _flush_segment(seg_start, t, votes_same, votes_swap)
            seg_start = t
            votes_same = votes_swap = 0
            continue

        # 1 detection → assign to closest
        if n_det == 1:
            src_idx = np.where(present)[0][0]
            dist_to_0 = np.hypot(x_raw[src_idx, t] - last_x[0], y_raw[src_idx, t] - last_y[0])
            dist_to_1 = np.hypot(x_raw[src_idx, t] - last_x[1], y_raw[src_idx, t] - last_y[1])
            if min(dist_to_0, dist_to_1) <= MAX_DIST_BETWEEN_FRAMES:
                dest_idx = 0 if dist_to_0 <= dist_to_1 else 1
                _assign_single_detection(t, src_idx, dest_idx)
                _update_votes(t)
            continue

        # 2 detections
        # compute distances and assignment costs
        inter_d = np.hypot(x_raw[0, t] - x_raw[1, t], y_raw[0, t] - y_raw[1, t])
        dx = x_raw[:, t][:, None] - last_x[None, :]
        dy = y_raw[:, t][:, None] - last_y[None, :]
        dist_mat = np.hypot(dx, dy)
        cost_same = dist_mat[0, 0] + dist_mat[1, 1]
        cost_swap = dist_mat[0, 1] + dist_mat[1, 0]
        min_dist_id0 = dist_mat[0].min()
        min_dist_id1 = dist_mat[1].min()

        # Define discontinuity conditions
        break_too_close = inter_d < MIN_INTER_SUBJ_DIST  # two detections are too close
        break_too_costly = min(cost_same, cost_swap) > MAX_SWAP_COST
        break_both_far = min(min_dist_id0, min_dist_id1) > MAX_DIST_BETWEEN_FRAMES  # both assignments >90px

        if break_too_close or (break_too_costly and break_both_far):
            _flush_segment(seg_start, t, votes_same, votes_swap)
            seg_start = t
            votes_same = votes_swap = 0

        # Re-evaluate after reset
        if break_too_close:  # keep original assignment
            _assign_full_frame(t, x_raw[:, t], y_raw[:, t])
            votes_same += 1
            continue

        # Re-evaluate after reset
        if break_too_costly:
            # If both assignments are too far, skip
            if break_both_far:
                continue
            dest_idx = 0 if min_dist_id0 <= min_dist_id1 else 1
            src_idx = int(np.argmin(dist_mat[dest_idx]))
            _assign_single_detection(t, src_idx, dest_idx)
            _update_votes(t)
            continue

        if cost_same <= cost_swap:
            x_vals = x_raw[:, t]
            y_vals = y_raw[:, t]
            votes_same += 1
        else:
            x_vals = x_raw[::-1, t]
            y_vals = y_raw[::-1, t]
            swapped_flags[t] = True
            votes_swap += 1
        _assign_full_frame(t, x_vals, y_vals)

    # 7) Flush the final segment
    _flush_segment(seg_start, T, votes_same, votes_swap)

    # 8) Build cleaned DataFrame
    cleaned = pd.DataFrame(
        {
            time_col: np.repeat(times, 2),
            "identity_name": np.tile(ids, T),
            "x": x_clean.ravel(order="F"),
            "y": y_clean.ravel(order="F"),
        }
    )

    # 9) Merge back with original data (drop old x, y)
    df2_noxy = df2.drop(columns=["x", "y"])
    result = (
        df2_noxy.merge(cleaned, on=[time_col, "identity_name"], how="right")
        .set_index(time_col)
        .sort_index()
    )

    # 10) Mark swapped frames in likelihood
    if "identity_likelihood" in result.columns:
        mask = result.index.isin(times[swapped_flags])
        result.loc[mask, "identity_likelihood"] = np.nan

    # 11) Final cleanup: drop any rows where x or y is NaN
    result = result.dropna(subset=["x", "y"], how="any")

    return result

### Load and prepare data

In [ ]:
# Load all periods for experiment
data = load_experiment_data(
    experiment=experiment,
    data_dir=data_dir,
    periods=['social'],
    data_types=['position', 'positiondenoised'],
)

social_position_df = data['social_position']
end_time = social_position_df.index.max()
start_time = end_time - pd.Timedelta(days=1)
social_position_df = social_position_df.loc[social_position_df.index > start_time]
social_position_df_CLEAN = data['social_position']
social_position_df_CLEAN = social_position_df_CLEAN.loc[social_position_df_CLEAN.index > start_time]
social_position_df_DENOISE = data['social_positiondenoised']
if not pd.api.types.is_datetime64_any_dtype(social_position_df_DENOISE.index):
    social_position_df_DENOISE.index = pd.to_datetime(social_position_df_DENOISE.index)
social_position_df_DENOISE = social_position_df_DENOISE.loc[social_position_df_DENOISE.index > start_time]

In [ ]:
# Clean the position data
key = {"experiment_name": experiment['name']}
region_df = (acquisition.EpochConfig.ActiveRegion & key).fetch(format="frame")
social_position_df_CLEAN = clean_swaps(social_position_df_CLEAN, region_df)
social_position_df_CLEAN.reset_index(inplace=True, drop=False)
# Sort by time for consistency
social_position_df_CLEAN.sort_values('time')
social_position_df_DENOISE.sort_values('time')

### Compare timestamps

In [ ]:
# 1) Align and round both time columns to the same resolution
ts_clean = pd.to_datetime(social_position_df_CLEAN['time']).dt.round('1ms')
ts_denoise = pd.to_datetime(social_position_df_DENOISE['time']).dt.round('1ms')

# 2) Extract unique timestamps and sort
unique_clean   = pd.Series(ts_clean.unique()).sort_values().reset_index(drop=True)
unique_denoise = pd.Series(ts_denoise.unique()).sort_values().reset_index(drop=True)

print(f"CLEAN   unique timestamps: {len(unique_clean):,}")
print(f"DENOISE unique timestamps: {len(unique_denoise):,}")

# 3) Compare via set‐difference
set_clean   = set(unique_clean)
set_denoise = set(unique_denoise)

only_in_clean   = sorted(set_clean - set_denoise)
only_in_denoise = sorted(set_denoise - set_clean)

print(f"Timestamps only in CLEAN:   {len(only_in_clean):,}")
print(f"Timestamps only in DENOISE: {len(only_in_denoise):,}")

# 4) Optionally, inspect mismatches
if only_in_clean or only_in_denoise:
    diff_df = pd.DataFrame({
        "only_in_clean":   only_in_clean   + [pd.NaT] * max(0, len(only_in_denoise)-len(only_in_clean)),
        "only_in_denoise": only_in_denoise + [pd.NaT] * max(0, len(only_in_clean)-len(only_in_denoise))
    })
    display(diff_df)
else:
    print("✅ CLEAN and DENOISE have exactly the same set of timestamps.")


### Compare content

In [ ]:
# 1) Align and round times
df_clean = social_position_df_CLEAN.copy()
df_denoise = social_position_df_DENOISE.copy()

df_clean['time']   = pd.to_datetime(df_clean['time']).dt.round('1ms')
df_denoise['time'] = pd.to_datetime(df_denoise['time']).dt.round('1ms')

# 2) Merge on time + identity_name
merged = pd.merge(
    df_clean,
    df_denoise,
    on=['time','identity_name'],
    how='inner',
    suffixes=('_clean','_denoise')
)

# 3) Compute differences
merged['dx']   = merged['x_clean'] - merged['x_denoise']
merged['dy']   = merged['y_clean'] - merged['y_denoise']
merged['dist'] = np.sqrt(merged['dx']**2 + merged['dy']**2)

# 4) Identify all differing rows
diff_mask = (merged['dx'] != 0) | (merged['dy'] != 0)
diff_df   = merged.loc[diff_mask, [
    'time', 'identity_name',
    'x_clean', 'y_clean',
    'x_denoise', 'y_denoise',
    'dx', 'dy', 'dist'
]].reset_index(drop=True)

# 5) Summary
n_diff   = len(diff_df)
n_total  = len(merged)
pct_diff = n_diff / n_total * 100

print(f"Matched rows:   {n_total:,}")
print(f"Differing rows: {n_diff:,} ({pct_diff:.3f}% mismatch)")

# 6) Display the full DataFrame of mismatches
diff_df.sort_values('time')

### Check for (timestamp, subject) duplicates (there should be none)

In [ ]:
# Inspect duplicates in both CLEAN and DENOISE

for label, df in [
    ("CLEAN", social_position_df_CLEAN),
    ("DENOISE", social_position_df_DENOISE)
]:
    print(f"\n=== {label} ===")
    # 1) Flag all rows whose (time, identity_name) is duplicated
    dup_mask = df.duplicated(subset=['time','identity_name'], keep=False)
    
    # 2) Subset to just those duplicates
    dup_df = df.loc[dup_mask].copy()
    
    # 3) Sort for readability
    dup_df = dup_df.sort_values(['time','identity_name'])
    
    # 4) Counts
    total_dupes = len(dup_df)
    unique_keys = dup_df.drop_duplicates(subset=['time','identity_name']).shape[0]
    print(f"Total duplicate rows: {total_dupes:,}")
    print(f"Unique (time,identity) pairs duplicated: {unique_keys:,}")
    
    # 5) Show a sample
    display(dup_df.sort_values('time'))
